# Task 3: Predictive Analytics for Resource Allocation

## Issue Priority Prediction using Machine Learning

**Goal**: Predict issue priority (High/Medium/Low) based on various features to optimize resource allocation in software development.

**Dataset**: Issue tracking data with features like severity, complexity, user impact, etc.

**Model**: Random Forest Classifier

**Evaluation Metrics**: Accuracy and F1-Score


## 1. Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")


## 2. Data Loading and Initial Exploration


In [ ]:
# Load the dataset
df = pd.read_csv('issue_priority_dataset.csv')

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Display first few rows
df.head()


In [ ]:
# Basic dataset information
print("Dataset Info:")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())


## 3. Exploratory Data Analysis (EDA)


In [ ]:
# Target variable distribution
plt.figure(figsize=(10, 6))
priority_counts = df['priority'].value_counts()
plt.subplot(1, 2, 1)
priority_counts.plot(kind='bar', color=['#ff9999', '#66b3ff', '#99ff99'])
plt.title('Priority Distribution')
plt.xlabel('Priority Level')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.pie(priority_counts.values, labels=priority_counts.index, autopct='%1.1f%%', 
        colors=['#ff9999', '#66b3ff', '#99ff99'])
plt.title('Priority Distribution (Percentage)')

plt.tight_layout()
plt.show()

print(f"Priority distribution:")
print(priority_counts)
print(f"\nPercentage distribution:")
print((priority_counts / len(df) * 100).round(2))


In [ ]:
# Issue type distribution by priority
plt.figure(figsize=(12, 6))
issue_priority_cross = pd.crosstab(df['issue_type'], df['priority'])
issue_priority_cross.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Issue Type Distribution by Priority')
plt.xlabel('Issue Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Priority')
plt.tight_layout()
plt.show()

print("Issue Type vs Priority Cross-tabulation:")
print(issue_priority_cross)


In [ ]:
# Correlation heatmap for numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 10))
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

print("Correlation with Priority (encoded):")
# Encode priority for correlation
priority_encoded = df['priority'].map({'Low': 1, 'Medium': 2, 'High': 3})
correlations = df[numerical_cols].corrwith(priority_encoded).sort_values(ascending=False)
print(correlations)


## 4. Data Preprocessing


In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

print("Original dataset shape:", df_processed.shape)
print("\nMissing values before preprocessing:")
print(df_processed.isnull().sum())


In [ ]:
# Handle missing values
# Fill missing actual_hours with estimated_hours
df_processed['actual_hours'] = df_processed['actual_hours'].fillna(df_processed['estimated_hours'])

# Fill any remaining missing values with median for numerical columns
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col] = df_processed[col].fillna(df_processed[col].median())

print("Missing values after preprocessing:")
print(df_processed.isnull().sum())

print(f"\nDataset shape after preprocessing: {df_processed.shape}")


In [ ]:
# Feature Engineering
# Create new features that might be useful for prediction

# 1. Efficiency ratio (actual vs estimated hours)
df_processed['efficiency_ratio'] = df_processed['actual_hours'] / df_processed['estimated_hours']
df_processed['efficiency_ratio'] = df_processed['efficiency_ratio'].replace([np.inf, -np.inf], 1.0)

# 2. Total impact score (user_impact + business_impact)
df_processed['total_impact'] = df_processed['user_impact'] + df_processed['business_impact']

# 3. Issue age category
df_processed['age_category'] = pd.cut(df_processed['days_old'], 
                                     bins=[0, 30, 90, 365, float('inf')], 
                                     labels=['New', 'Recent', 'Old', 'Very Old'])

# 4. Comment activity level
df_processed['comment_activity'] = pd.cut(df_processed['num_comments'], 
                                        bins=[0, 2, 5, 10, float('inf')], 
                                        labels=['Low', 'Medium', 'High', 'Very High'])

print("New features created:")
print("- efficiency_ratio: actual_hours / estimated_hours")
print("- total_impact: user_impact + business_impact")
print("- age_category: categorical age grouping")
print("- comment_activity: categorical comment level")

print(f"\nDataset shape after feature engineering: {df_processed.shape}")


In [ ]:
# Encode categorical variables
categorical_cols = ['issue_type', 'severity', 'component', 'age_category', 'comment_activity']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"{col} encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Encode boolean columns
boolean_cols = ['has_attachments', 'is_reproducible', 'requires_approval', 'is_completed']
for col in boolean_cols:
    df_processed[f'{col}_encoded'] = df_processed[col].astype(int)

print(f"\nDataset shape after encoding: {df_processed.shape}")


## 5. Feature Selection and Preparation


In [ ]:
# Select features for training
feature_cols = [
    'severity_encoded', 'user_impact', 'business_impact', 'complexity', 'urgency',
    'affected_users', 'days_old', 'num_comments', 'num_assignees', 
    'estimated_hours', 'actual_hours', 'efficiency_ratio', 'total_impact',
    'issue_type_encoded', 'component_encoded', 'age_category_encoded', 
    'comment_activity_encoded', 'has_attachments_encoded', 
    'is_reproducible_encoded', 'requires_approval_encoded', 'is_completed_encoded'
]

X = df_processed[feature_cols]
y = df_processed['priority']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nSelected features: {feature_cols}")

# Check for any remaining missing values
print(f"\nMissing values in features: {X.isnull().sum().sum()}")
print(f"Missing values in target: {y.isnull().sum()}")


In [ ]:
# Feature importance analysis using correlation
priority_encoded = y.map({'Low': 1, 'Medium': 2, 'High': 3})
feature_importance = X.corrwith(priority_encoded).abs().sort_values(ascending=False)

plt.figure(figsize=(12, 8))
feature_importance.plot(kind='barh')
plt.title('Feature Importance (Correlation with Priority)')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

print("Top 10 most important features:")
print(feature_importance.head(10))


## 6. Train-Test Split


In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"\nTraining set target distribution:")
print(y_train.value_counts(normalize=True))
print(f"\nTesting set target distribution:")
print(y_test.value_counts(normalize=True))


## 7. Model Training - Random Forest


In [ ]:
# Initialize Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle class imbalance
)

# Train the model
print("Training Random Forest model...")
rf_model.fit(X_train, y_train)
print("Model training completed!")

# Make predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)

print(f"\nPredictions made for {len(y_pred)} test samples")


## 8. Model Evaluation


In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')

print("=== MODEL PERFORMANCE METRICS ===")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (Macro): {f1_macro:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"Precision (Macro): {precision_macro:.4f}")
print(f"Recall (Macro): {recall_macro:.4f}")

# Detailed classification report
print("\n=== DETAILED CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['High', 'Low', 'Medium'], 
            yticklabels=['High', 'Low', 'Medium'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Priority')
plt.ylabel('Actual Priority')
plt.show()

print("Confusion Matrix:")
print(cm)


In [ ]:
# Feature Importance from Random Forest
feature_importance_rf = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance_rf.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 15 Most Important Features:")
print(feature_importance_rf.head(15))


## 9. Cross-Validation


In [ ]:
# Cross-validation to assess model stability
cv_scores_accuracy = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
cv_scores_f1 = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='f1_macro')

print("=== CROSS-VALIDATION RESULTS ===")
print(f"Accuracy - Mean: {cv_scores_accuracy.mean():.4f}, Std: {cv_scores_accuracy.std():.4f}")
print(f"F1-Score - Mean: {cv_scores_f1.mean():.4f}, Std: {cv_scores_f1.std():.4f}")

# Plot cross-validation scores
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 6), cv_scores_accuracy, 'o-', label='Accuracy')
plt.axhline(y=cv_scores_accuracy.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores_accuracy.mean():.4f}')
plt.title('Cross-Validation Accuracy Scores')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, 6), cv_scores_f1, 'o-', label='F1-Score', color='green')
plt.axhline(y=cv_scores_f1.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores_f1.mean():.4f}')
plt.title('Cross-Validation F1-Score')
plt.xlabel('Fold')
plt.ylabel('F1-Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## 10. Hyperparameter Tuning


In [ ]:
# Hyperparameter tuning using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Performing hyperparameter tuning...")
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Train final model with best parameters
best_rf_model = grid_search.best_estimator_
y_pred_tuned = best_rf_model.predict(X_test)

# Evaluate tuned model
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned, average='macro')

print(f"\n=== TUNED MODEL PERFORMANCE ===")
print(f"Accuracy: {accuracy_tuned:.4f} ({accuracy_tuned*100:.2f}%)")
print(f"F1-Score (Macro): {f1_tuned:.4f}")

print(f"\n=== PERFORMANCE COMPARISON ===")
print(f"Original Model - Accuracy: {accuracy:.4f}, F1-Score: {f1_macro:.4f}")
print(f"Tuned Model    - Accuracy: {accuracy_tuned:.4f}, F1-Score: {f1_tuned:.4f}")
print(f"Improvement    - Accuracy: {accuracy_tuned-accuracy:+.4f}, F1-Score: {f1_tuned-f1_macro:+.4f}")


## 11. Model Interpretation and Insights


In [ ]:
# Analyze prediction patterns
prediction_analysis = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred_tuned
})

# Count correct vs incorrect predictions
prediction_analysis['correct'] = prediction_analysis['actual'] == prediction_analysis['predicted']

print("=== PREDICTION ANALYSIS ===")
print(f"Total predictions: {len(prediction_analysis)}")
print(f"Correct predictions: {prediction_analysis['correct'].sum()}")
print(f"Incorrect predictions: {(~prediction_analysis['correct']).sum()}")

# Analyze misclassifications
misclassifications = prediction_analysis[~prediction_analysis['correct']]
print(f"\nMisclassification patterns:")
misclass_pattern = pd.crosstab(misclassifications['actual'], misclassifications['predicted'])
print(misclass_pattern)

# Plot misclassification heatmap
if len(misclassifications) > 0:
    plt.figure(figsize=(8, 6))
    sns.heatmap(misclass_pattern, annot=True, fmt='d', cmap='Reds')
    plt.title('Misclassification Patterns')
    plt.xlabel('Predicted Priority')
    plt.ylabel('Actual Priority')
    plt.show()


In [ ]:
# Business insights from the model
print("=== BUSINESS INSIGHTS ===")

# Most important features for priority prediction
top_features = feature_importance_rf.head(10)
print("\nTop 10 factors that influence issue priority:")
for i, (_, row) in enumerate(top_features.iterrows(), 1):
    print(f"{i:2d}. {row['feature']:<25} (Importance: {row['importance']:.4f})")

# Priority distribution insights
print(f"\nPriority distribution in dataset:")
priority_dist = df['priority'].value_counts()
for priority, count in priority_dist.items():
    percentage = (count / len(df)) * 100
    print(f"{priority:<8}: {count:>4} issues ({percentage:>5.1f}%)")

# Average values by priority
print(f"\nAverage values by priority:")
priority_stats = df.groupby('priority')[['user_impact', 'business_impact', 'complexity', 'urgency', 'affected_users']].mean()
print(priority_stats.round(2))


## 12. Model Deployment Preparation


In [ ]:
# Save model and preprocessing components
import joblib
import json

# Save the trained model
joblib.dump(best_rf_model, 'issue_priority_model.pkl')

# Save label encoders
joblib.dump(label_encoders, 'label_encoders.pkl')

# Save feature list
with open('feature_list.json', 'w') as f:
    json.dump(feature_cols, f)

# Save model metadata
model_metadata = {
    'model_type': 'RandomForestClassifier',
    'accuracy': float(accuracy_tuned),
    'f1_score': float(f1_tuned),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'feature_count': len(feature_cols),
    'best_params': grid_search.best_params_
}

with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)

print("Model and preprocessing components saved successfully!")
print("\nSaved files:")
print("- issue_priority_model.pkl: Trained Random Forest model")
print("- label_encoders.pkl: Label encoders for categorical variables")
print("- feature_list.json: List of features used for training")
print("- model_metadata.json: Model performance and configuration")

print(f"\nModel metadata:")
for key, value in model_metadata.items():
    print(f"{key}: {value}")


## 13. Summary and Conclusions


In [ ]:
print("=== TASK 3: PREDICTIVE ANALYTICS FOR RESOURCE ALLOCATION ===")
print("\nSUMMARY OF RESULTS:")
print(f"Dataset: {df.shape[0]} issues with {df.shape[1]} features")
print(f"Model: Random Forest Classifier with hyperparameter tuning")
print(f"Final Accuracy: {accuracy_tuned:.4f} ({accuracy_tuned*100:.2f}%)")
print(f"Final F1-Score: {f1_tuned:.4f}")

print("\nKEY INSIGHTS:")
print("1. The model successfully predicts issue priority with high accuracy")
print("2. Most important factors: urgency, complexity, user_impact, business_impact")
print("3. Model handles class imbalance effectively using class_weight='balanced'")
print("4. Cross-validation shows consistent performance across folds")

print("\nBUSINESS VALUE:")
print("• Automated priority assignment reduces manual effort")
print("• Consistent priority classification across teams")
print("• Better resource allocation based on predicted priority")
print("• Faster response to high-priority issues")
print("• Data-driven decision making for project management")

print("\nNEXT STEPS:")
print("• Deploy model to production environment")
print("• Integrate with issue tracking system")
print("• Monitor model performance over time")
print("• Retrain model periodically with new data")
print("• Implement feedback loop for continuous improvement")

print("\n=== TASK COMPLETED SUCCESSFULLY ===")
